# 19 - Building graphs by time of day

Every network built so far in this project collapses the whole schedule into a single static graph: a segment `u -> v` exists if some trip, at any hour, on any day, runs from `u` to `v`. That's a useful first model, but it hides the fact that a public-transport network isn't one network - it's a different network every few hours. A station that's a critical cut point at 03:00 might be one of many redundant alternatives at 08:00.

This notebook is part 1 of future research direction 1. It slices the schedule into five time windows and builds a separate graph for each one, in a single streaming pass over the 816 MB `stop_times.txt` file. It also brings `calendar.txt` into use - a GTFS file the project has never touched - so that each trip can be tagged as running on weekdays (Sunday-Thursday), Friday, Saturday, or some combination of these. That distinction matters a great deal in Israel, where most bus and rail service stops on Saturday, so a single graph of all service quietly averages a full weekday network together with an almost empty Saturday one.

Notebook 20 consumes the graphs written here to compare centrality and robustness across the windows.

The research question here: how much does the topology of Israel's public-transport network really change over the day and over the week - does the network shrink, break apart, or just thin out, and by how much?

## A caveat that applies to every number below

GTFS is a schedule. Everything counted in this notebook is planned trips and planned segment traversals. These are not ridership data, not vehicle occupancy, and not service that actually happened. A segment served by 40 planned buses in the morning peak might carry 4,000 passengers or 40; nothing in this feed can tell us which. Wherever the text says service volume, it means planned departures.

## Input

* `israel-public-transportation/stop_times.txt` - 816 MB, 15.7M rows, not tracked in git; downloaded on demand from Google Drive by the cell below (same file id as notebook 02).
* `israel-public-transportation/trips.txt` - maps `trip_id` to `service_id`.
* `israel-public-transportation/calendar.txt` - maps `service_id` to the weekday pattern it runs on.
* `outputs/nb/02_graph_construction/tables/nodes.csv` - station attributes (`stop_id, stop_name, lat, lon, region, metro`) used to decorate each window's graph.

## Output

Everything is written under `outputs/nb/19_time_of_day_graphs/`:

* `tables/window_summary.csv` - `window, start_hour, end_hour, trips, nodes, directed_edges, avg_degree, largest_component_share` (the contract other notebooks depend on).
* `tables/edges_<window>.csv` - one file per window, `from_stop, to_stop, trip_frequency`.
* `graph_<window>.pkl` - one undirected `networkx` graph, pickled, per window.
* `tables/window_summary_by_daytype.csv` - the same statistics broken out by weekday / Friday / Saturday service.
* `tables/hourly_service_volume.csv` - planned departures from stations and trip starts for each hour of the service day.
* `tables/service_calendar_summary.csv` - how many services and how many trips run on each day of the week.
* `tables/window_edge_overlap.csv` - Jaccard overlap of the edge sets for each pair of windows.
* `figures/service_volume_by_hour.png`, `figures/network_size_by_window.png`, `figures/connectivity_by_window.png`, `figures/weekday_vs_weekend_by_window.png`, `figures/window_edge_overlap.png`.
* `window_construction_summary.json` - all the key numbers plus streaming-pass statistics.

Nothing outside `outputs/nb/19_time_of_day_graphs/` is written.

## Notebooks that must run first

* `02_graph_construction` - for `tables/nodes.csv`. (It's also the notebook that first downloads `stop_times.txt`, though this notebook can download it itself.)

No further step is needed.

## 1. Initializing the working environment

The cell below lets the notebook run both on a local copy and on Google Colab. It defines `_ensure(...)`, which pip-installs only the packages that are actually missing (so re-running the notebook is cheap), and `find_repo_root()`, which climbs upward from the current directory looking for the GTFS folder and, if it isn't found, clones the repository into `/content`. It then sets `REPO`, `DATA`, and `OUT` and creates the notebook's output folder. Every later cell relies on these three paths, so this cell has to run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders, and tunable constants

We import the scientific toolkit and fix the folder layout: this notebook holds exactly one output folder, `outputs/nb/19_time_of_day_graphs/`, with the subfolders `tables/` and `figures/`.

All the expensive or judgment-call settings are gathered here, so a reviewer can change them in one place:

* `WINDOWS` is the time-slice definition, in the form `(name, start_hour, end_hour)` where `end_hour` is exclusive and wrapping around the day is allowed (`night` runs from 23:00 to 06:00). The five windows have to cover all 24 hours exactly once - the next cell asserts this, so you can edit this list freely and the notebook will complain immediately if the new definition creates a hole or an overlap.
* `PROGRESS_EVERY` sets how often the streaming pass prints progress. Cost note: the streaming pass reads all 15.7M rows of the 816 MB `stop_times.txt` file exactly once, and usually takes 4-8 minutes; this is by far the dominant component of the notebook's cost. It builds all the window counters at once, so re-reading the file for each window (which would cost 5x) never happens. Peak memory is a few hundred MB - the trip-to-service mapping (about 420k records) plus 15 edge counters of at most about 50k records each.
* `WRITE_WINDOW_PICKLES` can be set to `False` to skip writing the five `graph_<window>.pkl` files (a few MB each). Notebook 20 reads them, so leave it `True` unless you only need the tables.
* `FIG_DPI` and `TOP_N` affect only figure size and table length.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import csv, json, pickle, time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

# A handful of rows in stop_times.txt are very long; raise the csv field limit up front.
csv.field_size_limit(10_000_000)

STAGE = OUT / '19_time_of_day_graphs'    # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
# (name, start_hour, end_hour) with end_hour EXCLUSIVE; a window may wrap past midnight.
WINDOWS = [
    ('morning_peak',   6,  9),
    ('midday',         9, 15),
    ('afternoon_peak', 15, 19),
    ('evening',        19, 23),
    ('night',          23,  6),
]

PROGRESS_EVERY = 2_000_000     # rows between progress prints in the single streaming pass
WRITE_WINDOW_PICKLES = True    # write graph_<window>.pkl (notebook 20 needs these)
FIG_DPI = 150                  # figure resolution; drop to 90 for faster, smaller files
TOP_N = 15                     # rows shown in preview tables

print('pandas', pd.__version__, '| networkx', nx.__version__)
print('this stage :', STAGE)

## 3. Handling Hebrew labels

Station names in the Israeli GTFS feed are in Hebrew, and the preview tables and one figure below print them. Matplotlib doesn't implement Unicode's bidirectional algorithm, so right-to-left text comes out reversed and unreadable. The cell below applies a one-time monkey-patch to `matplotlib.text.Text.set_text` so that any string containing Hebrew characters is converted to display order with `python-bidi` before it's drawn, and picks a font that actually contains Hebrew glyphs (Arial on Windows, DejaVu Sans everywhere else). The cell is idempotent - re-running it won't stack patch on patch. All other text in the notebook is in English, as the submission requires.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Locating outputs from earlier stages

This notebook needs only one earlier output: `nodes.csv` from notebook 02, which provides the station names, coordinates, and regions we attach to each window graph. Stage folders are identified by their two-digit prefix rather than the full slug, so a folder with a different name (`02_graph_construction` versus `02_graphs`) is still recognized. If the folder or file is missing we raise a `FileNotFoundError` that names exactly which notebook to run first, instead of failing later with an opaque `KeyError`.

In [ ]:
# --- Stage resolution helpers ---------------------------------------------
def stage_dir(prefix, notebook_hint):
    """Return the output folder whose name starts with `prefix` (e.g. '02')."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            'No stage folder starting with ' + repr(prefix) + ' under ' + str(OUT) +
            ' - run notebook ' + notebook_hint + ' first.'
        )
    return matches[0]


def load_stage_table(prefix, filename, notebook_hint, **read_kwargs):
    """Load a CSV from an earlier stage, searching the stage folder recursively."""
    folder = stage_dir(prefix, notebook_hint)
    direct = folder / filename
    if direct.exists():
        path = direct
    else:
        found = sorted(folder.rglob(filename))
        if not found:
            raise FileNotFoundError(
                filename + ' not found anywhere under ' + str(folder) +
                ' - run notebook ' + notebook_hint + ' first; it writes ' + filename + '.'
            )
        path = found[0]
    df = pd.read_csv(path, encoding='utf-8-sig', **read_kwargs)
    print('loaded ' + filename + ': ' + format(len(df), ',') + ' rows  <-  ' + str(path))
    return df


def require_raw(filename):
    """Return the path of a raw GTFS file, or raise a clear error."""
    path = DATA / filename
    if not path.exists():
        raise FileNotFoundError(
            str(path) + ' is missing - the raw GTFS feed must be present in ' + str(DATA) + '.'
        )
    return path


nodes_df = load_stage_table('02', 'nodes.csv', '02_graph_construction',
                            dtype={'stop_id': str})
nodes_df.head(3)

## 5. Downloading `stop_times.txt`

The `stop_times.txt` file is 816 MB and is deliberately not tracked in git. The cell below downloads it from Google Drive on demand, using exactly the same file id as notebook 02, and does nothing if the file already exists. Nothing else in the feed needs downloading - `trips.txt` and `calendar.txt` are small and live in the repository.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 6. Time windows and decoding GTFS times

Two things are defined here.

The hour-to-window lookup table. `WINDOWS` is expanded into a 24-entry table mapping each hour of the service day to exactly one window. The expansion asserts that the windows tile the day - no hour belongs to two windows, no hour is left out - so a change to the `WINDOWS` constant will fail loudly rather than silently drop segments.

Decoding GTFS times. GTFS times are measured from service midnight, not wall-clock midnight, and are allowed to exceed 24 hours: `25:30:00` means 01:30 the next calendar morning, but still belongs to the previous service day. About 1.1% of the rows in this feed are of this kind. Passing such a string to `datetime.strptime` raises `ValueError: unconverted data remains` and crashes the pass, so we don't use `datetime` at all - `gtfs_seconds` splits on `:` and returns `h*3600 + m*60 + s`. To place a 25:30 departure in the `night` window we take the hour modulo 24, and that's the intended semantics: a 25:30 bus is a 01:30 bus in terms of how the network looks at that moment. The assertions below lock this behavior in place.

In [ ]:
# --- Expand WINDOWS into an hour -> window lookup --------------------------
def window_hours(start, end):
    """Hours covered by [start, end); wraps past midnight when end <= start."""
    span = (end - start) % 24 or 24
    return [(start + i) % 24 for i in range(span)]


WINDOW_NAMES = [name for name, _, _ in WINDOWS]
WINDOW_INDEX = {name: i for i, name in enumerate(WINDOW_NAMES)}

_hour_owner = {}
for _name, _s, _e in WINDOWS:
    for _h in window_hours(_s, _e):
        if _h in _hour_owner:
            raise ValueError(
                'WINDOWS overlap: hour ' + str(_h) + ' is claimed by both ' +
                _hour_owner[_h] + ' and ' + _name + '.'
            )
        _hour_owner[_h] = _name
_gaps = [h for h in range(24) if h not in _hour_owner]
if _gaps:
    raise ValueError('WINDOWS leave hours uncovered: ' + str(_gaps))

# Plain python list -> fastest possible lookup inside the 15.7M-row loop.
HOUR_TO_WIDX = [WINDOW_INDEX[_hour_owner[h]] for h in range(24)]
HOUR_TO_WINDOW = {h: _hour_owner[h] for h in range(24)}


def gtfs_seconds(value):
    """Parse an HH:MM:SS GTFS time into seconds since SERVICE midnight.

    Hours may be >= 24 (25:30:00 = 01:30 on the next calendar day, same service day).
    Never use datetime/strptime here - it raises on hours >= 24.
    Returns None for blank or malformed values.
    """
    if not value:
        return None
    parts = value.split(':')
    if len(parts) != 3:
        return None
    try:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    except ValueError:
        return None


# Sanity checks on the two tricky cases.
assert gtfs_seconds('05:10:00') == 5 * 3600 + 10 * 60
assert gtfs_seconds('25:30:00') == 25 * 3600 + 30 * 60
assert (gtfs_seconds('25:30:00') // 3600) % 24 == 1
assert gtfs_seconds('') is None and gtfs_seconds('nonsense') is None

print('hour -> window map:')
for name, s, e in WINDOWS:
    hrs = window_hours(s, e)
    print('  ' + name.ljust(16) + str(s).rjust(2) + ':00 -> ' + str(e).rjust(2) +
          ':00  (' + str(len(hrs)) + ' hours: ' + ', '.join(str(h) for h in hrs) + ')')

## 7. `calendar.txt` - which day of the week does each service run on?

This is the first time the project touches `calendar.txt`. In GTFS, each trip carries a `service_id`, and `calendar.txt` gives each service seven boolean columns (`sunday` ... `saturday`) plus a validity date range. The Israeli work week runs from Sunday to Thursday; Friday is a short day where service tapers off through the afternoon, and on Saturday the great majority of bus and rail traffic doesn't run at all.

So we reduce each service to three independent booleans:

* `runs_weekday` - runs on at least one of Sunday through Thursday;
* `runs_friday`;
* `runs_saturday`.

These are not mutually exclusive, and that's deliberate: a service with the pattern `1111110` really does run both on weekdays and on Friday, and it would be wrong to force it into a single category. As a result, a trip may be counted in more than one day-type column later, and the day-type columns do not sum to a total. The cell prints the day-pattern census so this is visible rather than assumed.

One honest caveat: for a small number of services all seven day flags are set to `0`. In a full GTFS feed these services would all be driven by `calendar_dates.txt` (explicitly added service dates), but this feed does not include `calendar_dates.txt`, so for those services we simply don't know which day they run on. They're reported separately as `unknown` and are still included in the all-service graphs.

In [ ]:
# --- Read calendar.txt and reduce every service to day-type flags ----------
DAY_COLS = ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday']
WEEKDAY_COLS = ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday']

# Bit codes, so the streaming pass can carry the day-type in a single small int.
BIT_WEEKDAY, BIT_FRIDAY, BIT_SATURDAY = 1, 2, 4

calendar_path = require_raw('calendar.txt')
service_code = {}          # service_id -> bit code
day_service_counts = Counter()
pattern_counts = Counter()

with open(calendar_path, encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        flags = {d: str(row.get(d, '0')).strip() == '1' for d in DAY_COLS}
        code = 0
        if any(flags[d] for d in WEEKDAY_COLS):
            code |= BIT_WEEKDAY
        if flags['friday']:
            code |= BIT_FRIDAY
        if flags['saturday']:
            code |= BIT_SATURDAY
        service_code[str(row['service_id']).strip()] = code
        for d in DAY_COLS:
            if flags[d]:
                day_service_counts[d] += 1
        pattern_counts[''.join('1' if flags[d] else '0' for d in DAY_COLS)] += 1

n_services = len(service_code)
n_unknown_services = sum(1 for c in service_code.values() if c == 0)
print('services in calendar.txt      : ' + format(n_services, ','))
print('services with no day flag set : ' + format(n_unknown_services, ',') +
      '  (no calendar_dates.txt in this feed -> day of week unknown)')
print()
print('Most common weekly patterns (Sun Mon Tue Wed Thu Fri Sat):')
for pat, cnt in pattern_counts.most_common(10):
    print('  ' + ' '.join(pat) + '   ' + format(cnt, '>8,') + ' services')

## 8. `trips.txt` - attaching a day-type to each trip

`stop_times.txt` only knows `trip_id`, so before the big pass we build a `trip_id -> day-type bit code` dictionary by joining `trips.txt` to the service codes from the previous cell. This dictionary (about 420k records) is the only large object held in memory besides the edge counters.

The cell also produces the trip-weighted view of the week, which is far more informative than the service-weighted one: services are administrative entities of wildly different sizes, whereas trips are actual planned vehicle journeys. The Saturday figure printed here is the central evidence that breaking service out by day-type matters.

In [ ]:
# --- Map every trip_id to its day-type bit code ----------------------------
trips_path = require_raw('trips.txt')
trip_code = {}
trips_missing_service = 0

with open(trips_path, encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sid = str(row.get('service_id', '')).strip()
        if sid not in service_code:
            trips_missing_service += 1
        trip_code[str(row['trip_id']).strip()] = service_code.get(sid, 0)

n_trips_total = len(trip_code)
trip_day_counts = {
    'weekday_sun_thu': sum(1 for c in trip_code.values() if c & BIT_WEEKDAY),
    'friday': sum(1 for c in trip_code.values() if c & BIT_FRIDAY),
    'saturday': sum(1 for c in trip_code.values() if c & BIT_SATURDAY),
    'unknown': sum(1 for c in trip_code.values() if c == 0),
}

print('trips in trips.txt                 : ' + format(n_trips_total, ','))
print('trips whose service_id is missing  : ' + format(trips_missing_service, ','))
print()
print('Scheduled trips by day of week (a trip can count in more than one row):')
for label, cnt in trip_day_counts.items():
    share = 100.0 * cnt / n_trips_total if n_trips_total else 0.0
    print('  ' + label.ljust(18) + format(cnt, '>9,') + '   ' + format(share, '5.1f') + '% of all trips')

service_calendar = pd.DataFrame(
    [{'day': d, 'services_running': day_service_counts.get(d, 0)} for d in DAY_COLS]
)
service_calendar['is_israeli_weekend'] = service_calendar['day'].isin(['friday', 'saturday'])
service_calendar.to_csv(TABLES / 'service_calendar_summary.csv', index=False, encoding='utf-8-sig')
service_calendar

## 9. The single streaming pass over `stop_times.txt`

This is the expensive cell, and this is the most significant design decision: one pass, all counters at once.

The file is sorted by `(trip_id, stop_sequence)` - notebook 02 verifies this across the whole file - so two consecutive rows carrying the same `trip_id` are consecutive stops of the same trip, and hence one directed segment. For each such pair we assign the segment to a window by the departure time at the origin stop (the earlier row), which is the moment the vehicle actually crosses the segment. Every counter below is updated in that same iteration:

* `edges_all[w]` - segment-traversal counts per window, across all services;
* `edges_wd[w]`, `edges_fr[w]`, `edges_sa[w]` - the same, restricted to trips whose service runs on weekdays / Friday / Saturday;
* `hour_*` - planned departures from stations per hour of the service day, and per hour and day-type;
* `trip_start_*` - trip starts per hour (the first row in each trip block);
* `trip_window_mask` - a 5-bit mask per trip recording which windows that trip touched, so a trip crossing the 09:00 boundary is counted in both windows it serves.

Memory use is `O(|E| * windows + |T|)`, never `O(rows)`: no row is stored. A `defaultdict(int)` keyed by `(from_stop, to_stop)` holds at most a few tens of thousands of records per window.

Cost: about 15.7M rows, usually 4-8 minutes on a laptop. Reading the file once per window would cost five times as much, and that's exactly what this structure prevents.

Rows with an unparseable or empty `departure_time` are counted and skipped, rather than guessing their value.

In [ ]:
def stream_window_edges(path, trip_code, progress_every=PROGRESS_EVERY):
    """One pass over stop_times.txt, building every per-window counter at once.

    A segment (prev_stop -> stop) is assigned to the window containing the
    DEPARTURE time at prev_stop. Times are seconds since service midnight and the
    hour is taken modulo 24, so a 25:30 departure lands in the window covering 01:00.
    """
    n_w = len(WINDOW_NAMES)
    edges_all = [defaultdict(int) for _ in range(n_w)]
    edges_wd = [defaultdict(int) for _ in range(n_w)]
    edges_fr = [defaultdict(int) for _ in range(n_w)]
    edges_sa = [defaultdict(int) for _ in range(n_w)]

    hour_all = [0] * 24
    hour_wd = [0] * 24
    hour_fr = [0] * 24
    hour_sa = [0] * 24
    start_all = [0] * 24
    start_wd = [0] * 24
    start_fr = [0] * 24
    start_sa = [0] * 24

    trip_window_mask = defaultdict(int)

    rows_read = 0
    segments = 0
    self_loops_skipped = 0
    bad_times = 0
    rows_past_midnight = 0
    trips_seen = 0
    trips_not_in_trips_txt = set()
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index('trip_id')
        si = header.index('stop_id')
        di = header.index('departure_time')

        prev_trip = None
        prev_stop = None
        prev_secs = None
        code = 0

        for row in reader:
            rows_read += 1
            trip = row[ti]
            stop = row[si]
            secs = gtfs_seconds(row[di])
            if secs is None:
                bad_times += 1
            elif secs >= 86400:
                rows_past_midnight += 1

            new_trip = trip != prev_trip
            if new_trip:
                trips_seen += 1
                if trip in trip_code:
                    code = trip_code[trip]
                else:
                    code = 0
                    if len(trips_not_in_trips_txt) < 50:
                        trips_not_in_trips_txt.add(trip)

            is_wd = bool(code & BIT_WEEKDAY)
            is_fr = bool(code & BIT_FRIDAY)
            is_sa = bool(code & BIT_SATURDAY)

            if secs is not None:
                h = (secs // 3600) % 24
                hour_all[h] += 1
                if is_wd:
                    hour_wd[h] += 1
                if is_fr:
                    hour_fr[h] += 1
                if is_sa:
                    hour_sa[h] += 1
                if new_trip:
                    start_all[h] += 1
                    if is_wd:
                        start_wd[h] += 1
                    if is_fr:
                        start_fr[h] += 1
                    if is_sa:
                        start_sa[h] += 1

            if not new_trip and prev_stop is not None and prev_secs is not None:
                if prev_stop != stop:
                    w = HOUR_TO_WIDX[(prev_secs // 3600) % 24]
                    key = (prev_stop, stop)
                    edges_all[w][key] += 1
                    if is_wd:
                        edges_wd[w][key] += 1
                    if is_fr:
                        edges_fr[w][key] += 1
                    if is_sa:
                        edges_sa[w][key] += 1
                    trip_window_mask[trip] |= (1 << w)
                    segments += 1
                else:
                    self_loops_skipped += 1

            prev_trip, prev_stop, prev_secs = trip, stop, secs

            if progress_every and rows_read % progress_every == 0:
                uniq = sum(len(d) for d in edges_all)
                print('    ' + format(rows_read, ',') + ' rows | ' + format(uniq, ',') +
                      ' window-segments | ' + format(time.time() - t0, ',.0f') + 's')

    stats = {
        'stop_times_rows': rows_read,
        'trip_blocks': trips_seen,
        'segments_assigned': segments,
        'self_loops_skipped': self_loops_skipped,
        'rows_with_unparseable_time': bad_times,
        'rows_with_hour_ge_24': rows_past_midnight,
        'trips_absent_from_trips_txt_sample': sorted(trips_not_in_trips_txt)[:5],
        'elapsed_seconds': round(time.time() - t0, 1),
    }
    hourly = {'all': hour_all, 'weekday': hour_wd, 'friday': hour_fr, 'saturday': hour_sa}
    starts = {'all': start_all, 'weekday': start_wd, 'friday': start_fr, 'saturday': start_sa}
    edges = {'all': edges_all, 'weekday': edges_wd, 'friday': edges_fr, 'saturday': edges_sa}
    return edges, hourly, starts, trip_window_mask, stats


print('Streaming ' + STOP_TIMES.name + ' (~15.7M rows, one pass, expect 4-8 minutes) ...')
edge_sets, hourly_counts, start_counts, trip_window_mask, stream_stats = stream_window_edges(
    STOP_TIMES, trip_code)

print()
print('Streaming statistics:')
for k, v in stream_stats.items():
    print('  ' + k + ': ' + str(v))

## 10. Service volume by hour of the service day

The first thing to look at is the raw shape of the schedule. For each of the 24 hours of the service day, we tabulate how many planned departures from stations occur (one per row in `stop_times.txt` with a usable time) and how many trips start. Both are broken out by day-type. Remember that the day-type columns overlap - a service running Sunday through Friday contributes to both `weekday` and `friday` - so they aren't an exhaustive partition and should be read as how much service exists on a representative day of that type, not as a share of the total.

The figure shades the five windows so you can judge the window boundaries against the schedule's actual demand profile, and plots weekday, Friday, and Saturday volume on the same axis. Saturday should look very close to the floor.

In [ ]:
# --- Hourly service volume table ------------------------------------------
hourly_df = pd.DataFrame({
    'hour': list(range(24)),
    'window': [HOUR_TO_WINDOW[h] for h in range(24)],
    'stop_departures_all': hourly_counts['all'],
    'stop_departures_weekday': hourly_counts['weekday'],
    'stop_departures_friday': hourly_counts['friday'],
    'stop_departures_saturday': hourly_counts['saturday'],
    'trips_starting_all': start_counts['all'],
    'trips_starting_weekday': start_counts['weekday'],
    'trips_starting_friday': start_counts['friday'],
    'trips_starting_saturday': start_counts['saturday'],
})
hourly_df.to_csv(TABLES / 'hourly_service_volume.csv', index=False, encoding='utf-8-sig')


def contiguous_runs(hours):
    """Collapse a list of hours into (first, last) runs, for shading a wrapped window."""
    hs = sorted(hours)
    runs, start, prev = [], hs[0], hs[0]
    for h in hs[1:]:
        if h == prev + 1:
            prev = h
        else:
            runs.append((start, prev))
            start = prev = h
    runs.append((start, prev))
    return runs


band_palette = sns.color_palette('pastel', len(WINDOWS))
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

for ax, (col_prefix, ylabel) in zip(
        axes,
        [('stop_departures', 'scheduled stop departures'),
         ('trips_starting', 'trips starting')]):
    for i, (name, s, e) in enumerate(WINDOWS):
        for a, b in contiguous_runs(window_hours(s, e)):
            ax.axvspan(a - 0.5, b + 0.5, color=band_palette[i], alpha=0.45, zorder=0)
    for suffix, style in [('all', '-'), ('weekday', '--'), ('friday', '-.'), ('saturday', ':')]:
        ax.plot(hourly_df['hour'], hourly_df[col_prefix + '_' + suffix],
                style, marker='o', markersize=3, linewidth=1.8, label=suffix, zorder=3)
    ax.set_ylabel(ylabel)
    ax.legend(title='service runs on', ncol=4, loc='upper left')

for i, (name, s, e) in enumerate(WINDOWS):
    for a, b in contiguous_runs(window_hours(s, e)):
        axes[0].text((a + b) / 2.0, axes[0].get_ylim()[1] * 0.02, name,
                     ha='center', va='bottom', fontsize=8, rotation=90, zorder=4)

axes[1].set_xlabel('hour of the service day (GTFS hours >= 24 folded back modulo 24)')
axes[1].set_xticks(range(24))
axes[0].set_title('Scheduled service volume by hour of day, with the five time windows shaded\n'
                  '(SCHEDULED trips - not observed ridership)')
plt.tight_layout()
plt.savefig(FIGURES / 'service_volume_by_hour.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

hourly_df

## 11. Building one graph per window

Each window's directed segment counter is now converted into an undirected graph, exactly the way notebook 02 builds the static network: the two travel directions of a segment merge into a single edge whose weight is the sum of the two directed traversal counts. Station attributes (`stop_name`, `lat`, `lon`, `region`, `metro`) are joined in from stage 02's `nodes.csv`, so downstream notebooks get a self-describing graph.

For each window we then measure:

* `trips` - trips contributing at least one segment to the window. A trip crossing a window boundary is counted in every window it touches, so this column deliberately sums to more than the number of unique trips.
* `nodes` - stations with at least one segment in the window. A station served only by trips outside the window simply doesn't appear in that window's graph.
* `directed_edges` - unique ordered segments, following stage 02's `edges.csv` convention.
* `avg_degree` - average undirected degree, `2m/n`.
* `largest_component_share` - the share of the window's nodes that lie in its largest connected component. This is the fragmentation measure: a value well below 1 means the network is already split apart before any attack is even simulated.

The per-window `edges_<window>.csv` files and the `graph_<window>.pkl` pickles are written here; they're the contract notebook 20 depends on.

In [ ]:
# --- Node attributes from stage 02 ----------------------------------------
def _num(value):
    """Coerce to float; return None for blanks, NaN or non-numeric input."""
    if value is None:
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


NODE_ATTR = {}
for rec in nodes_df.to_dict('records'):
    NODE_ATTR[str(rec.get('stop_id'))] = {
        'stop_name': _txt(rec.get('stop_name')),
        'lat': _num(rec.get('lat')),
        'lon': _num(rec.get('lon')),
        'region': _txt(rec.get('region')),
        'metro': _txt(rec.get('metro')),
    }
_DEFAULT_ATTR = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}


def build_undirected(edge_dict):
    """Undirected projection of a directed segment counter; weights of both directions summed."""
    G = nx.Graph()
    for (u, v), w in edge_dict.items():
        if G.has_edge(u, v):
            G[u][v]['weight'] += w
        else:
            G.add_edge(u, v, weight=w)
    for n in G.nodes():
        G.nodes[n].update(NODE_ATTR.get(n, _DEFAULT_ATTR))
    return G


def graph_stats(G):
    """nodes, undirected edges, avg degree, largest-component share."""
    n = G.number_of_nodes()
    m = G.number_of_edges()
    if n == 0:
        return {'nodes': 0, 'undirected_edges': 0, 'avg_degree': 0.0,
                'components': 0, 'largest_component_size': 0,
                'largest_component_share': 0.0}
    comps = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    return {
        'nodes': n,
        'undirected_edges': m,
        'avg_degree': round(2.0 * m / n, 3),
        'components': len(comps),
        'largest_component_size': comps[0],
        'largest_component_share': round(comps[0] / n, 4),
    }


# --- Trips per window, from the 5-bit masks collected during the pass ------
trips_per_window = [0] * len(WINDOW_NAMES)
trips_per_window_by_daytype = {dt: [0] * len(WINDOW_NAMES)
                               for dt in ('weekday', 'friday', 'saturday')}
for trip, mask in trip_window_mask.items():
    code = trip_code.get(trip, 0)
    for w in range(len(WINDOW_NAMES)):
        if mask & (1 << w):
            trips_per_window[w] += 1
            if code & BIT_WEEKDAY:
                trips_per_window_by_daytype['weekday'][w] += 1
            if code & BIT_FRIDAY:
                trips_per_window_by_daytype['friday'][w] += 1
            if code & BIT_SATURDAY:
                trips_per_window_by_daytype['saturday'][w] += 1

# --- Build, save and summarise the all-service window graphs ---------------
window_graphs = {}
summary_rows = []
for w, (name, s, e) in enumerate(WINDOWS):
    ed = edge_sets['all'][w]
    G = build_undirected(ed)
    window_graphs[name] = G
    st = graph_stats(G)

    edges_out = pd.DataFrame(
        [{'from_stop': u, 'to_stop': v, 'trip_frequency': int(c)} for (u, v), c in ed.items()]
    ).sort_values('trip_frequency', ascending=False)
    edges_out.to_csv(TABLES / ('edges_' + name + '.csv'), index=False, encoding='utf-8-sig')

    if WRITE_WINDOW_PICKLES:
        with open(STAGE / ('graph_' + name + '.pkl'), 'wb') as fh:
            pickle.dump(G, fh)

    summary_rows.append({
        'window': name,
        'start_hour': s,
        'end_hour': e,
        'trips': trips_per_window[w],
        'nodes': st['nodes'],
        'directed_edges': len(ed),
        'avg_degree': st['avg_degree'],
        'largest_component_share': st['largest_component_share'],
    })
    print(name.ljust(16) + 'nodes ' + format(st['nodes'], '>6,') +
          ' | directed edges ' + format(len(ed), '>7,') +
          ' | avg degree ' + format(st['avg_degree'], '5.2f') +
          ' | LCC share ' + format(st['largest_component_share'], '5.3f') +
          ' | components ' + format(st['components'], '>5,'))

window_summary = pd.DataFrame(summary_rows)
window_summary.to_csv(TABLES / 'window_summary.csv', index=False, encoding='utf-8-sig')
window_summary

## 12. The same windows, broken out by day-type

The table above averages a full Tuesday together with an almost service-free Saturday. Here we rebuild those same five windows three more times - restricted to trips whose service runs on weekdays, on Friday, and on Saturday - and report exactly the same statistics. These graphs are not saved as pickles (notebook 20 works from the all-service windows); the point of this section is diagnostic: to show how much of the static network is really a product of weekdays.

A reminder: the three day-types overlap by definition, and services with an `unknown` day (no flags set in `calendar.txt`) don't appear in any of the three, though they do appear in the all-service graphs above.

In [ ]:
# --- Window statistics per day type ---------------------------------------
daytype_rows = []
for daytype in ('weekday', 'friday', 'saturday'):
    for w, (name, s, e) in enumerate(WINDOWS):
        ed = edge_sets[daytype][w]
        st = graph_stats(build_undirected(ed))
        daytype_rows.append({
            'window': name,
            'day_type': daytype,
            'start_hour': s,
            'end_hour': e,
            'trips': trips_per_window_by_daytype[daytype][w],
            'nodes': st['nodes'],
            'directed_edges': len(ed),
            'avg_degree': st['avg_degree'],
            'components': st['components'],
            'largest_component_share': st['largest_component_share'],
        })

daytype_summary = pd.DataFrame(daytype_rows)
daytype_summary.to_csv(TABLES / 'window_summary_by_daytype.csv', index=False, encoding='utf-8-sig')

pivot_nodes = daytype_summary.pivot(index='window', columns='day_type', values='nodes')
pivot_nodes = pivot_nodes.reindex(WINDOW_NAMES)
print('Stations reachable in each window, by day type:')
print(pivot_nodes.to_string())
print()

saturday_total = daytype_summary.loc[daytype_summary['day_type'] == 'saturday', 'directed_edges'].sum()
weekday_total = daytype_summary.loc[daytype_summary['day_type'] == 'weekday', 'directed_edges'].sum()
if weekday_total:
    print('Saturday segment coverage is ' +
          format(100.0 * saturday_total / weekday_total, '.1f') +
          '% of the weekday figure (summed over windows).')

daytype_summary

## 13. How network size changes between windows

Three views of the same result. The first panel shows how many stations and how many unique segments exist in each window - the size of the network. The second shows average degree and the largest-component share - the shape of the network: whether the off-peak network is just a smaller copy of the peak network, or a structurally different, more fragmented one. The third figure repeats the station count by day-type, and there the Saturday effect stands out most sharply.

In [ ]:
# --- Figure: network size across windows ----------------------------------
order = WINDOW_NAMES
ws = window_summary.set_index('window').reindex(order)
x = np.arange(len(order))

fig, ax1 = plt.subplots(figsize=(11, 6))
ax1.bar(x - 0.2, ws['nodes'], width=0.4, label='stations (nodes)', color='#4C72B0')
ax1.set_ylabel('stations in the window graph', color='#4C72B0')
ax1.tick_params(axis='y', labelcolor='#4C72B0')
ax2 = ax1.twinx()
ax2.bar(x + 0.2, ws['directed_edges'], width=0.4, label='directed segments', color='#DD8452')
ax2.set_ylabel('distinct directed segments', color='#DD8452')
ax2.tick_params(axis='y', labelcolor='#DD8452')
ax2.grid(False)
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace('_', ' ') for n in order])
ax1.set_title('Network size by time window (scheduled service, all day types)')
for i, (n, e) in enumerate(zip(ws['nodes'], ws['directed_edges'])):
    ax1.text(i - 0.2, n, format(int(n), ','), ha='center', va='bottom', fontsize=8)
    ax2.text(i + 0.2, e, format(int(e), ','), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / 'network_size_by_window.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

# --- Figure: connectivity across windows ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(x, ws['avg_degree'], color='#55A868')
axes[0].set_xticks(x)
axes[0].set_xticklabels([n.replace('_', ' ') for n in order], rotation=20, ha='right')
axes[0].set_ylabel('average undirected degree')
axes[0].set_title('Average degree by window')
for i, v in enumerate(ws['avg_degree']):
    axes[0].text(i, v, format(v, '.2f'), ha='center', va='bottom', fontsize=9)

axes[1].bar(x, ws['largest_component_share'], color='#C44E52')
axes[1].set_ylim(0, 1.05)
axes[1].set_xticks(x)
axes[1].set_xticklabels([n.replace('_', ' ') for n in order], rotation=20, ha='right')
axes[1].set_ylabel('share of stations in the largest component')
axes[1].set_title('Fragmentation by window')
for i, v in enumerate(ws['largest_component_share']):
    axes[1].text(i, v, format(v, '.3f'), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'connectivity_by_window.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

# --- Figure: weekday vs Friday vs Saturday --------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
width = 0.26
colors = {'weekday': '#4C72B0', 'friday': '#DD8452', 'saturday': '#C44E52'}
for j, metric in enumerate(['nodes', 'directed_edges']):
    piv = daytype_summary.pivot(index='window', columns='day_type', values=metric).reindex(order)
    for k, dt in enumerate(['weekday', 'friday', 'saturday']):
        axes[j].bar(x + (k - 1) * width, piv[dt], width=width, label=dt, color=colors[dt])
    axes[j].set_xticks(x)
    axes[j].set_xticklabels([n.replace('_', ' ') for n in order], rotation=20, ha='right')
    axes[j].set_ylabel(metric.replace('_', ' '))
    axes[j].set_title(metric.replace('_', ' ') + ' by window and day type')
    axes[j].legend(title='service runs on')
plt.tight_layout()
plt.savefig(FIGURES / 'weekday_vs_weekend_by_window.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

## 14. Do the windows contain the same network, just sparser?

Size alone doesn't answer the interesting question. Two windows could each have 40,000 segments and share almost none of them (a genuinely different network) or share almost all of them (the same network at a different intensity). We measure this directly with the Jaccard index of the undirected edge sets for each pair of windows, `|A and B| / |A or B|`, plus the containment measure of the smaller window in the larger one, which shows how much of the night network is simply a subset of the morning peak.

This is a cheap computation - set operations on at most about 50k pairs per window.

In [ ]:
# --- Pairwise edge-set overlap between windows ----------------------------
undirected_sets = {
    name: {tuple(sorted((u, v))) for (u, v) in edge_sets['all'][w].keys()}
    for w, (name, _, _) in enumerate(WINDOWS)
}

overlap_rows = []
for a in WINDOW_NAMES:
    for b in WINDOW_NAMES:
        A, B = undirected_sets[a], undirected_sets[b]
        inter = len(A & B)
        union = len(A | B)
        overlap_rows.append({
            'window_a': a,
            'window_b': b,
            'edges_a': len(A),
            'edges_b': len(B),
            'shared_edges': inter,
            'jaccard': round(inter / union, 4) if union else 0.0,
            'share_of_a_covered_by_b': round(inter / len(A), 4) if A else 0.0,
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(TABLES / 'window_edge_overlap.csv', index=False, encoding='utf-8-sig')

jac = overlap_df.pivot(index='window_a', columns='window_b', values='jaccard').reindex(
    index=WINDOW_NAMES, columns=WINDOW_NAMES)
cov = overlap_df.pivot(index='window_a', columns='window_b',
                       values='share_of_a_covered_by_b').reindex(
    index=WINDOW_NAMES, columns=WINDOW_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(jac, annot=True, fmt='.2f', cmap='viridis', vmin=0, vmax=1, ax=axes[0],
            cbar_kws={'label': 'Jaccard index'})
axes[0].set_title('Edge-set similarity between windows (Jaccard)')
sns.heatmap(cov, annot=True, fmt='.2f', cmap='magma', vmin=0, vmax=1, ax=axes[1],
            cbar_kws={'label': 'share of row window covered by column window'})
axes[1].set_title('Containment: how much of window A also exists in window B')
for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIGURES / 'window_edge_overlap.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()

jac

## 15. The busiest segments in the morning peak and the night window

A quick qualitative check that the windows behave as expected: the most heavily served segments in the morning peak should look like dense urban arteries, and the night window should be dominated by a much shorter list of intercity or airport-type links. The station names are in Hebrew and are shown right-to-left thanks to the bidi patch installed earlier.

In [ ]:
# --- Top segments per window (sanity check) -------------------------------
def top_segments(window_name, k=TOP_N):
    w = WINDOW_INDEX[window_name]
    rows = []
    for (u, v), c in edge_sets['all'][w].items():
        rows.append({
            'from_stop': u,
            'from_name': NODE_ATTR.get(u, _DEFAULT_ATTR)['stop_name'],
            'to_stop': v,
            'to_name': NODE_ATTR.get(v, _DEFAULT_ATTR)['stop_name'],
            'trip_frequency': c,
        })
    return (pd.DataFrame(rows)
            .sort_values('trip_frequency', ascending=False)
            .head(k)
            .reset_index(drop=True))


print('=== Busiest segments, morning_peak ===')
display(top_segments('morning_peak'))
print('=== Busiest segments, night ===')
display(top_segments('night'))

## 16. Writing the stage summary

Everything a later notebook or reader might want as a single number is gathered into `window_construction_summary.json`: the window definition actually used, the streaming-pass statistics (rows read, segments assigned, rows with after-midnight hours, run time), the per-window statistics, the trip census by day-type, and the explicit caveats. The final log confirms which files were written and their sizes.

In [ ]:
# --- Stage summary --------------------------------------------------------
summary = {
    'stage': '19_time_of_day_graphs',
    'source': 'GTFS stop_times.txt + trips.txt + calendar.txt (SCHEDULED service, not ridership)',
    'windows': [{'window': n, 'start_hour': s, 'end_hour': e,
                 'hours': window_hours(s, e)} for n, s, e in WINDOWS],
    'segment_assignment_rule': 'departure time at the ORIGIN stop of the segment, hour taken modulo 24',
    'streaming_pass': stream_stats,
    'calendar': {
        'services_total': int(n_services),
        'services_with_no_day_flag': int(n_unknown_services),
        'trips_total': int(n_trips_total),
        'trips_by_day_type_overlapping': {k: int(v) for k, v in trip_day_counts.items()},
        'calendar_dates_txt_present': (DATA / 'calendar_dates.txt').exists(),
    },
    'per_window': {
        r['window']: {k: (float(r[k]) if k in ('avg_degree', 'largest_component_share')
                          else int(r[k]))
                      for k in ('trips', 'nodes', 'directed_edges',
                                'avg_degree', 'largest_component_share')}
        for r in summary_rows
    },
    'caveats': [
        'All counts are SCHEDULED trips from the GTFS timetable. They are not observed '
        'ridership, boardings, or vehicle occupancy; nothing in this feed measures demand.',
        'A trip that crosses a window boundary is counted in every window it touches, so the '
        'trips column sums to more than the number of distinct trips.',
        'Day types (weekday / friday / saturday) are not mutually exclusive: a service running '
        'Sunday-Friday is counted under both weekday and friday.',
        'Services with all seven day flags set to 0 have an unknown day of week because this '
        'feed ships no calendar_dates.txt; they are excluded from the day-type tables but '
        'included in the all-service window graphs.',
        'GTFS hours >= 24 are folded modulo 24, so a 25:30 departure is treated as 01:30 for '
        'the purpose of window assignment.',
    ],
}

with open(STAGE / 'window_construction_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Files written under ' + str(STAGE) + ':')
for p in sorted(STAGE.rglob('*')):
    if p.is_file():
        print('  ' + str(p.relative_to(STAGE)).ljust(46) +
              format(p.stat().st_size / 1024, '>10,.0f') + ' KB')

## Conclusions

Read these against the numbers actually printed above - the wording below describes what the analysis can and can't support, not a conclusion decided in advance.

1. The static network of notebooks 02-13 is a union, not a snapshot. No single moment in the week looks like the graph the rest of the project analyzes. The union of all windows contains every segment running at any hour on any day; each window graph on its own is strictly smaller than it, both in station count and in segment count. So any robustness claim made about the static graph is a claim about the most connected version of the network, which is the best case.

2. Off-peak degradation is mostly thinning, and partly genuine fragmentation. Compare `avg_degree` against `largest_component_share` across the windows in `window_summary.csv`. Where the largest-component share drops, the network isn't just running fewer buses - it has actually broken into pieces that can't reach each other at that hour, which is a stronger claim than low frequency.

3. The night window is a different network, not a smaller one. The Jaccard heatmap quantifies this: adjacent daytime windows overlap heavily (they're the same arteries at different intensities), whereas `night` shares a relatively small share of its edge set with the peak hours and is mostly contained in them rather than similar to them.

4. Saturday is the largest structural effect in the whole feed. The Saturday columns in `window_summary_by_daytype.csv` are a small fraction of the weekday columns. Averaging the week into one graph, as every earlier notebook does, therefore mixes a full-service network with an almost absent one. Any conclusion about equity or accessibility drawn from the static graph implicitly assumes weekday service.

5. What this notebook does not show. These are planned trips. A segment with a high `trip_frequency` is one the schedule serves often, and that's a proxy for - not a measurement of - importance to passengers. This feed has no ridership, occupancy, or reliability data, so a low-frequency segment serving a population with no alternatives and a high-frequency segment that has alternatives can't be told apart here. Notebook 21 tackles this gap with a population-based demand proxy, which is also a proxy.

6. Two honest limitations of the construction itself. (a) Window membership is decided by the departure time at the segment's origin stop, so a long intercity trip starting at 08:50 is attributed entirely to the morning peak even though most of it happens after 09:00; at window widths of 3-6 hours this affects only a small minority of segments, but it's a real approximation. (b) Because this feed has no `calendar_dates.txt`, services with all seven week flags zero can't be assigned a day of the week at all; they're reported as `unknown` and not quietly folded into the weekday data.

Handoff to notebook 20: `graph_<window>.pkl` and `tables/edges_<window>.csv` are the per-window networks; `tables/window_summary.csv` is the contract table. Notebook 20 recomputes centrality and simulates targeted removal within each window, to test whether the set of critical stations is stable over the day or whether criticality itself is time-dependent.